In [1]:
from bs4 import BeautifulSoup
import cloudscraper
import pandas as pd
from io import StringIO

In [ ]:
def extract_matchs(season):

    url = f'https://fbref.com/en/squads/206d90db/{season}/all_comps/Barcelona-Stats-All-Competitions'
    scraper = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows", "mobile": False}
    )

    # 1️⃣ Descargar HTML real (saltando Cloudflare)
    r = scraper.get(url)
    r.raise_for_status()

    # 2️⃣ Parsear HTML
    soup = BeautifulSoup(r.text, "lxml")

    # 3️⃣ Seleccionar tabla correcta por ID (fixture table)
    table = soup.find("table", {"id": "matchlogs_for"})

    # 4️⃣ Convertir tabla a DataFrame
    df = pd.read_html(StringIO(str(table)))[0]

    # 5️⃣ Extraer links Match Report desde el HTML (no desde pandas)
    rows = table.find("tbody").find_all("tr")

    match_urls = []

    for row in rows:
        cell = row.find("td", {"data-stat": "match_report"})
        if cell is None:
            match_urls.append(None)
            continue

        a = cell.find("a")
        if a and "href" in a.attrs:
            match_urls.append("https://fbref.com" + a["href"])
        else:
            match_urls.append(None)

    # 6️⃣ Agregar columna al DataFrame
    df["match_report_url"] = match_urls
    df.to_csv(f'/home/fjordan/Documentos/Proyectos/Personales/proyectos-Python/Analisis Lamine-Pedri/data/raw/score_fixtures_{season}.csv', index=False)
    return df


In [3]:
season='2025-2026'
df = extract_matchs(season)

HTTPError: 403 Client Error: Forbidden for url: https://fbref.com/en/squads/206d90db/2025-2026/all_comps/Barcelona-Stats-All-Competitions

In [4]:
def clean_matchs(df):

    df = df.copy()
    df = df[['match_report_url','Date','Comp', 'Venue', 'Opponent', 'GF', 'GA', 'xG', 'xGA', 'Poss']]
    df['match_id'] = df['match_report_url'].str.extract(r'/matches/([^/]+)/')
    df['season'] = season
    df = df[['match_id', 'match_report_url', 'Date', 'season', 'Comp', 'Opponent', 'Venue', 'GF', 'GA', 'xG', 'xGA', 'Poss']]
    df.rename(columns={'match_report_url':'match_url',
                               'Date':'date',
                               'Comp':'competition',
                               'Opponent':'opponent',
                               'Venue':'home_away',
                               'GF':'goals_for',
                               'GA':'goals_against',
                               'xG':'xG_for',
                               'xGA':'xG_against',
                               'Poss':'posesion'}, inplace=True)
    df.to_csv(f'/home/fjordan/Documentos/Proyectos/Personales/proyectos-Python/Analisis Lamine-Pedri/data/clean/score_fixtures_clean_{season}.csv', index=False)
    return df

In [10]:
score_fixtures_2025_26 = clean_matchs(df)

In [6]:
score_fixtures_2025_26 = pd.read_csv('/home/fjordan/Documentos/Proyectos/Personales/proyectos-Python/Analisis Lamine-Pedri/data/clean/score_fixtures_clean_2025-2026.csv')

score_fixtures_2025_26.head()

,match_id,match_url,date,season,competition,opponent,home_away,goals_for,goals_against,xG_for,xG_against,posesion
0,64fa5074,https://fbref.com/en/matches/64fa5074/Mallorca...,2025-08-16,2025-2026,La Liga,Mallorca,Away,3.0,0.0,2.1,0.2,70.0
1,28628853,https://fbref.com/en/matches/28628853/Levante-...,2025-08-23,2025-2026,La Liga,Levante,Away,3.0,2.0,1.9,1.8,81.0
2,38c8e28a,https://fbref.com/en/matches/38c8e28a/Rayo-Val...,2025-08-31,2025-2026,La Liga,Rayo Vallecano,Away,1.0,1.0,1.8,1.8,56.0
3,2ed529d8,https://fbref.com/en/matches/2ed529d8/Barcelon...,2025-09-14,2025-2026,La Liga,Valencia,Home,6.0,0.0,3.9,0.1,71.0
4,53708b73,https://fbref.com/en/matches/53708b73/Newcastl...,2025-09-18,2025-2026,Champions Lg,eng Newcastle Utd,Away,2.0,1.0,1.1,1.2,63.0


In [7]:
score_fixtures_2024_25 = pd.read_csv('/home/fjordan/Documentos/Proyectos/Personales/proyectos-Python/Analisis Lamine-Pedri/data/clean/score_fixtures_clean_2024-2025.csv')
score_fixtures_2024_25.head()

,match_id,match_url,date,season,competition,opponent,home_away,goals_for,goals_against,xG_for,xG_against,posesion
0,6f0fac43,https://fbref.com/en/matches/6f0fac43/Valencia...,2024-08-17,2024-2025,La Liga,Valencia,Away,2,1,3.2,1.0,63
1,87bc9bba,https://fbref.com/en/matches/87bc9bba/Barcelon...,2024-08-24,2024-2025,La Liga,Athletic Club,Home,2,1,1.8,1.0,64
2,f8fc00c7,https://fbref.com/en/matches/f8fc00c7/Rayo-Val...,2024-08-27,2024-2025,La Liga,Rayo Vallecano,Away,2,1,1.4,0.4,64
3,3ed1b8ff,https://fbref.com/en/matches/3ed1b8ff/Barcelon...,2024-08-31,2024-2025,La Liga,Valladolid,Home,7,0,4.7,0.5,70
4,f166ffc9,https://fbref.com/en/matches/f166ffc9/Girona-B...,2024-09-15,2024-2025,La Liga,Girona,Away,4,1,1.9,1.3,55
